## Imports

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from abm_simulator import ABMSimulator
from flood_adapt import FloodAdapt
from flood_adapt.config.config import Settings

## Config

In [2]:
# FloodAdapt databse inputs
# DATA_DIR = Path(r"c:\Users\athanasi\Github\Database\Working_Databases\Charleston\4_FloodAdapt\Database")
DATA_DIR = Path(r"c:\Users\winter_ga\Offline_data\FloodAdapt-WorkingDatabase\Charleston\4_FloodAdapt\Database")

site = "charleston_beta_release"

# Monte Carlo input parameters for event sequences
timestep = 1  # timestep of Monte Carlo in years
sim_time = 30  # length of simulation in years
no_seq = 20  # number of event sequences to generate
seed = 42  # random seed for reproducibility

# SLR scenario to evaluate damages over time
slr_scenario_name = "NOAA High" # This needs to be a scenario in the FloodAdapt database

# lookup table
ds_impacts = xr.open_dataset("lookup_table_charleston.nc")

In [3]:
times = 2020 + np.linspace(0, sim_time, int(sim_time / timestep) + 1)
times

array([2020., 2021., 2022., 2023., 2024., 2025., 2026., 2027., 2028.,
       2029., 2030., 2031., 2032., 2033., 2034., 2035., 2036., 2037.,
       2038., 2039., 2040., 2041., 2042., 2043., 2044., 2045., 2046.,
       2047., 2048., 2049., 2050.])

## Initialize FloodAdapt Database

In [4]:
settings = Settings(
    DATABASE_ROOT=DATA_DIR,
    DATABASE_NAME=site,
)
fa = FloodAdapt(database_path=settings.database_path)
slr_values = [fa.interp_slr(slr_scenario=slr_scenario_name, year=time) for time in times]

2025-12-16 03:12:56 PM - FloodAdapt.Database - INFO - Initializing database to charleston_beta_release at c:\users\winter_ga\offline_data\floodadapt-workingdatabase\charleston\4_floodadapt\database


In [5]:
ds_impacts

<xarray.Dataset> Size: 427MB
Dimensions:       (object_id: 61858, slr: 4, strategy: 2, event: 4)
Coordinates:
  * object_id     (object_id) <U1598 395MB '14048' '67199' ... '70438' '70444'
  * slr           (slr) int32 16B 0 1 2 3
  * strategy      (strategy) <U16 128B 'no_measures' 'floodproof_all_0'
  * event         (event) <U10 160B 'subevent_1' 'subevent_2' ... 'subevent_4'
Data variables:
    inun_depth    (object_id, slr, strategy, event) float64 16MB ...
    total_damage  (object_id, slr, strategy, event) float64 16MB ...

In [6]:
from abm_simulator import ABMSimulator  

# Instantiate the simulator (event sequence and damage lookup are handled internally)
abm = ABMSimulator(
    ds_impacts=ds_impacts,
    times=times,
    slr_values=slr_values,
    no_seq=no_seq,
    damage_threshold=0.3,  # You can change this threshold as needed
    seed=seed
)


In [7]:
# Run the simulation
# abm.run("floor ")


In [ ]:
def calculate_damage_history(self, floodproofing: bool, method: str = 'linear'):
    """
    Shared logic for calculating damage history and per-event damage.
    If floodproofing is True, applies floodproofing logic; otherwise, always uses 'no_measures'.
    Returns:
        damage_history: [sequence, household, time] array
        damage_history_per_event: [sequence, household, time, event] array
        floodproofed: [sequence, household, time] boolean array (None if floodproofing is False)
    """
    n_events = len(self.event_names)
    event_names_list = list(self.event_names)
    damage_history = np.zeros((self.no_seq, self.n_households, self.time_steps))
    damage_history_per_event = np.zeros((self.no_seq, self.n_households, self.time_steps, n_events))
    floodproofed = np.zeros((self.no_seq, self.n_households, self.time_steps), dtype=bool) if floodproofing else None

    # full matrix lookups for no measures and floodproofing all (n_objects, n_events, n_slr_values)
    damage_matrix_no_measures = self.slr_damage_lookup(self.slr_values, event_names_list, 'no_measures', method="linear")
    if floodproofing:
        damage_matrix_floodproofing_all = self.slr_damage_lookup(self.slr_values, event_names_list, 'floodproof_all_0', method="linear")

    for seq_idx in range(self.no_seq):
        if floodproofing:
            print(f"Evaluating sequence {seq_idx+1}/{self.no_seq}...")
        else:
            print(f"[BASELINE] Evaluating sequence {seq_idx+1}/{self.no_seq}...")
        is_floodproofed = np.zeros(self.n_households, dtype=bool)
        for ti in range(self.time_steps):
            year_events = self.sequences[seq_idx][ti]
            total_damage = np.zeros(self.n_households)
            year_event_damage = np.zeros((self.n_households, n_events))
            for event in year_events:
                if event in event_names_list:
                    event_idx = event_names_list.index(event)
                    damages = damage_matrix_no_measures[:, event_idx, ti]
                    if floodproofing: # apply floodproofing if applicable
                        damages_floodproofing_all = damage_matrix_floodproofing_all[:, event_idx, ti]
                        damages = np.where(is_floodproofed, damages_floodproofing_all, damages)
                    year_event_damage[:, event_idx] = damages
                    total_damage += damages
            damage_history[seq_idx, :, ti] = total_damage
            damage_history_per_event[seq_idx, :, ti, :] = year_event_damage
            if floodproofing:
                floodproofed[seq_idx, :, ti] = is_floodproofed
                # Vectorized floodproofing decision
                not_floodproofed = ~is_floodproofed
                with_pot_dmg = self.max_pot_dmg > 0
                threshold_exceeded = np.zeros(self.n_households, dtype=bool)
                valid = not_floodproofed & with_pot_dmg
                threshold_exceeded[valid] = (total_damage[valid] / self.max_pot_dmg[valid]) > self.damage_threshold
                is_floodproofed = is_floodproofed | threshold_exceeded
    return damage_history, damage_history_per_event, floodproofed


event_names_list = list(abm.event_names)

damage_history, damage_history_per_event, floodproofed = calculate_damage_history(abm,floodproofing=True, method="linear")

Evaluating sequence 1/20...


UnboundLocalError: cannot access local variable 'damages' where it is not associated with a value

In [ ]:
abm.plot_event_damage_timeseries(1)

In [ ]:
abm.plot_total_damage_statistics()